In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

### Introduction

In this experiment, we extend the Graph Attention Network (GAT) from 4 layers to 6 layers to study the effect of increased depth.

Deeper GAT models allow nodes to aggregate information from larger neighborhoods through multiple attention-based message passing steps. However, increasing depth can lead to **oversmoothing**, where node representations become increasingly similar and less discriminative.

In [3]:
!pip install torch-geometric

In [4]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


### Loading Dataset 

In [5]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


In [6]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)
print(data.y.unique())

torch.Size([13752, 767])
torch.Size([2, 491722])
torch.Size([13752])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


### Loading the train test validation split

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/train_idx.pt")
val_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/val_idx.pt")
test_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/test_idx.pt")
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)

In [8]:

data = data.to(device)

In [9]:
print(len(train_idx), len(val_idx), len(test_idx))

9626 2063 2063


### GAT model Architecture (6 layers)

In [14]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8, dropout=0.6):
        super().__init__()
        
        self.dropout = dropout

        # Layer 1
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)

        # Hidden layers
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout)
        self.conv3 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout)
        self.conv4 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout)
        self.conv5 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout)

        # Output layer
        self.conv6 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv4(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv5(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv6(x, edge_index)

        return x

In [15]:
def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index)

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()

        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [16]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)

        pred = out[:batch.batch_size].argmax(dim=1)

        correct += (pred == batch.y[:batch.batch_size]).sum().item()
        total += batch.batch_size

    return correct / total

In [17]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [18]:
from sklearn.metrics import f1_score

@torch.no_grad()
def test_with_f1(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)
        pred = out[:batch.batch_size].argmax(dim=1)

        all_preds.append(pred.cpu())
        all_labels.append(batch.y[:batch.batch_size].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    return acc, macro_f1, micro_f1

### Experiment Pipeline

This function manages the end-to-end training and evaluation process.

- Initializes the model using the given configuration
- Constructs NeighborLoaders for mini-batch training
- Trains the model across epochs with early stopping based on validation performance
- Saves the best-performing model
- Evaluates final performance on the test set using Accuracy and F1 scores

In [19]:
def run_experiment(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = 50 if not run_test else 100

    
    model = GAT(
        in_channels=data.num_features,
        hidden_channels=config["hidden_dim"],
        out_channels=dataset.num_classes,
        heads=config["heads"],
        dropout=config["dropout"]
    ).to(device)

    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=5e-4
    )

    # loaders (same)
    train_loader = NeighborLoader(
        data,
        input_nodes=train_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=128,
        shuffle=True
    )

    val_loader = NeighborLoader(
        data,
        input_nodes=val_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    best_val = 0
    patience = 10
    counter = 0
    save_model = run_test

    for epoch in range(1, epochs + 1):
        loss = train(model, train_loader, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        #if run_test:
         #print(f"Epoch {epoch:03d} | Train Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            counter = 0

            if save_model:
                
                torch.save(model.state_dict(), f"/kaggle/working/best_modelGAT_{seed}.pt")
        else:
            counter += 1

        if counter >= patience:
            break

    if not run_test:
        return best_val

    # load best model
    model.load_state_dict(
        torch.load(f"/kaggle/working/best_modelGAT_{seed}.pt", map_location=device)
    )

    test_loader = NeighborLoader(
        data,
        input_nodes=test_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    acc, macro_f1, micro_f1 = test_with_f1(model, test_loader, device)

    return acc, macro_f1, micro_f1

In [21]:
best_config= {'hidden_dim': 8, 'lr': 0.001, 'heads': 8, 'dropout': 0.5, 'num_neighbors': [20, 10]}

### Final Training

In [ ]:

acc_list = []
micro_f1_list = []
macro_f1_list=[]
for seed in range(10):
    acc, macro_f1, micro_f1 = run_experiment(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")


Seed 0: Acc=0.8400, macro_F1=0.8116,micro_F1=0.8400
Seed 1: Acc=0.8696, macro_F1=0.8547,micro_F1=0.8696
Seed 2: Acc=0.8434, macro_F1=0.7665,micro_F1=0.8434
Seed 3: Acc=0.8381, macro_F1=0.7769,micro_F1=0.8381
Seed 4: Acc=0.8080, macro_F1=0.7255,micro_F1=0.8080
Seed 5: Acc=0.8303, macro_F1=0.7547,micro_F1=0.8303
Seed 6: Acc=0.8391, macro_F1=0.7532,micro_F1=0.8391
Seed 7: Acc=0.8342, macro_F1=0.7425,micro_F1=0.8342
Seed 8: Acc=0.8580, macro_F1=0.8253,micro_F1=0.8580
Seed 9: Acc=0.8013, macro_F1=0.7243,micro_F1=0.8013


### Final Results

In [23]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)

print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")


Final Results:
Accuracy: 0.8362 ± 0.0193
Macro F1: 0.7735 ± 0.0415
Micro F1: 0.8362 ± 0.0193
